# Porto Taxi Project – Stage 1 Demonstration & Exploration
This notebook demonstrates the cleaned data, basic statistics, and interactive map visualizations.

In [1]:
import json
import pandas as pd
import folium

# Load data report
with open('../output/cleaning_report.json') as f:
    report = json.load(f)

print('--- Cleaning Report Summary ---')
print(json.dumps(report['summary'], indent=2))

# Load cleaned dataset sample
df = pd.read_parquet('../output/cleaned_trips.parquet')
print(f'Cleaned Dataset Shape: {df.shape}')
df.head()

/Users/omriliberty/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


--- Cleaning Report Summary ---
{
  "final_clean_trips": 1622765,
  "unique_taxis": 442,
  "distance_percentiles_km": {
    "p25": 2.3932,
    "p50": 3.9579,
    "p75": 6.3798,
    "p90": 10.6236,
    "p95": 13.0512,
    "p99": 617.4314
  }
}
Cleaned Dataset Shape: (1622765, 20)


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,POLYLINE,coordinates,num_points,duration_sec,distance_km,start_lng,start_lat,end_lng,end_lat,avg_speed_kmh,trip_datetime,hour_of_day,day_of_week
0,1372636858620000589,C,,,20000589,1372636858,A,"[[-8.618643,41.141412],[-8.618499,41.141376],[...","[{'lng': -8.618643, 'lat': 41.141412}, {'lng':...",23,345,2.6510,-8.618643,41.141412,-8.630838,41.154489,27.66,2013-07-01 03:00:58,3,2
1,1372637303620000596,B,,7,20000596,1372637303,A,"[[-8.639847,41.159826],[-8.640351,41.159871],[...","[{'lng': -8.639847, 'lat': 41.159826}, {'lng':...",19,285,3.4563,-8.639847,41.159826,-8.665740,41.170671,43.66,2013-07-01 03:08:23,3,2
2,1372637091620000337,C,,,20000337,1372637091,A,"[[-8.645994,41.18049],[-8.645949,41.180517],[-...","[{'lng': -8.645994, 'lat': 41.18049}, {'lng': ...",29,435,4.8143,-8.645994,41.180490,-8.687268,41.178087,39.84,2013-07-01 03:04:51,3,2
3,1372636965620000231,C,,,20000231,1372636965,A,"[[-8.615502,41.140674],[-8.614854,41.140926],[...","[{'lng': -8.615502, 'lat': 41.140674}, {'lng':...",26,390,5.5696,-8.615502,41.140674,-8.578224,41.160717,51.41,2013-07-01 03:02:45,3,2
4,1372637210620000456,C,,,20000456,1372637210,A,"[[-8.57952,41.145948],[-8.580942,41.145039],[-...","[{'lng': -8.57952, 'lat': 41.145948}, {'lng': ...",36,540,3.4445,-8.579520,41.145948,-8.603973,41.142816,22.96,2013-07-01 03:06:50,3,2


In [2]:
# Summary statistics of extracted features
df[['num_points', 'duration_sec', 'distance_km', 'avg_speed_kmh']].describe()

,num_points,duration_sec,distance_km,avg_speed_kmh
count,1.622765e+06,1.622765e+06,1.622765e+06,1.622765e+06
mean,4.806912e+01,7.210368e+02,5.240210e+00,2.591977e+01
std,3.828978e+01,5.743468e+02,5.041352e+00,1.254127e+01
min,2.000000e+00,3.000000e+01,8.000000e-04,9.000000e-02
25%,2.900000e+01,4.350000e+02,2.433300e+00,1.756000e+01
50%,4.100000e+01,6.150000e+02,3.947700e+00,2.312000e+01
75%,5.800000e+01,8.700000e+02,6.450100e+00,3.132000e+01
max,3.585000e+03,5.377500e+04,6.174314e+02,1.333000e+02


In [3]:
# Interactive Map of Sample Taxi Trajectories
m = folium.Map(location=[41.1579, -8.6291], zoom_start=13, tiles='CartoDB positron')

sample_df = df.sample(1000, random_state=42)
for idx, row in sample_df.reset_index().iterrows():
    coords = row['coordinates']
    if coords is None or len(coords) < 2:
        continue
    lat_lngs = [[float(p['lat']), float(p['lng'])] for p in coords]
    folium.PolyLine(lat_lngs, color='blue', weight=2.5, opacity=0.7).add_to(m)
    folium.CircleMarker(lat_lngs[0], radius=3, color='green', fill=True).add_to(m)
    folium.CircleMarker(lat_lngs[-1], radius=3, color='red', fill=True).add_to(m)

m